# Setup

In [ ]:
import numpy as np
import polars as pl
import polars.selectors as cs

import torch
from torch import nn

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
from sentence_transformers.base.modules import Transformer, Dense
from sentence_transformers.sentence_transformer.modules import Pooling

import wandb
from datasets import load_dataset, load_dataset_builder
import huggingface_hub
from peft import LoraConfig, TaskType

from globalvars import *

In [ ]:
huggingface_hub.login()

# Load dataset

## Data preproc

In [ ]:
print(load_dataset_builder(HF_DATASET).info.features)

def preprocess_dset(df, with_nans=False):
    '''
    preprocess dataset 1 in polars

    input columns:
        trait: trait name (one of BIG5_TRAITS)
        level: 1 to 5 score
        description: personality trait description

    output columns:
        sentence: (unchanged)
        label: length-5 array of scores
            all scores are 0 except the target class which ranges from -1 to 1
        trait: (unchanged)
    '''
    SOFT_CLASS_WT = 0.7 # weight given to 2/5 and 4/5 ratings
    df = df.with_columns(
        pl.col('level') # 1 to 5
        .replace_strict([1, 2, 3, 4, 5], [-1.0, -SOFT_CLASS_WT, 0.0, SOFT_CLASS_WT, 1.0]) # -1 to 1
        .alias('score')
    )
    if with_nans:
        df = df.with_columns(
            (pl.col('trait') == trait).cast(float)
            .replace(0, np.nan)
            .mul(pl.col('score')) # -1 to 1 for this trait, nan for other traits
            .add(1).truediv(2) # 0 to 1 for this trait, nan for other traits
            .alias(f'trait_{trait}')
            for trait in BIG5_TRAITS
        )
    else:
        df = df.with_columns(
            (pl.col('trait') == trait).cast(int)
            .mul(pl.col('score')) # -1 to 1 for this trait, 0 for other traits
            .add(1).truediv(2) # 0 to 1 for this trait, 0.5 for other traits
            .alias(f'trait_{trait}')
            for trait in BIG5_TRAITS
        )
    df = (
        df.with_columns(label=pl.concat_arr(cs.starts_with('trait_')))
        .rename({'description': 'sentence'})
        .select('sentence', 'label', 'trait')
    )
    return df


ds = (
    load_dataset(HF_DATASET, split='all')
    .with_format('polars')
    .map(preprocess_dset, batched=True)
    .with_format(None)
    .class_encode_column('trait') # pyright: ignore[reportAttributeAccessIssue]
    .train_test_split(test_size=0.1, stratify_by_column='trait') # pyright: ignore[reportAttributeAccessIssue]
    .remove_columns(['trait'])
)

# ds_with_nans = (
#     load_dataset(HF_DATASET, split='all')
#     .with_format('polars')
#     .map(lambda d: preprocess_dset(d, with_nans=True), batched=True)
#     .with_format(None)
#     .class_encode_column('trait')
#     .train_test_split(test_size=0.1, stratify_by_column='trait')
#     .remove_columns(['trait'])
# )

# Define the sentence-transformers model

In [ ]:
def build_model(peft_config=None):
    '''Instantiate a new untrained model'''
    # Start with the pretrained base model (DistilBERT)
    word_embedding_module = Transformer(BASE_MODEL, max_seq_length=512)


    # Add a pooling layer
    # 'cls' pooling uses the CLS token which is the last hidden state's 1st token
    pooling_module = Pooling(
        word_embedding_module.get_embedding_dimension(),
        pooling_mode='cls'
    )

    # Add a dense layer that outputs a size-5 vector
    dense_module = Dense(
        in_features=word_embedding_module.get_embedding_dimension(),
        out_features=5,
        activation_function=nn.Identity() # Logits are returned directly
    )

    model = SentenceTransformer(
        modules=[word_embedding_module, pooling_module, dense_module],
        model_card_data=SentenceTransformerModelCardData(
            language=["en", "es"],
            model_name="distilBERT-based Big-5 personality scorer",
            model_id=MODEL,
            train_datasets=[{'id': HF_DATASET}],
            eval_datasets=[{'id': HF_DATASET}],
            task_name='feature extraction',
            tags=['feature-extraction'],
        )
    )
    if peft_config:
        model.add_adapter(peft_config)
    return model


In [ ]:
class MultiLabelBCEWithLogitsLoss(nn.Module):
    def __init__(self, model: SentenceTransformer):
        super(MultiLabelBCEWithLogitsLoss, self).__init__()
        self.model = model
        self.criterion = nn.BCEWithLogitsLoss()
        # self.criterion = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, sentence_features: list[dict[str, torch.Tensor]], labels: torch.Tensor):
        # sentence_features is a list containing the feature dict for each input text
        # For standard classification/regression tasks in ST, we take the first element
        outputs = self.model(sentence_features[0])
        logits = outputs['sentence_embedding']

        # labels expected shape: (batch_size, 5)
        return self.criterion(logits, labels.float())
        # return self.criterion(logits, labels.float()).nanmean()


# Hyperparameters

## Parameter sweep

In [ ]:
sweep_config = {
    'method': 'bayes',
    'metric': {
        'name': 'eval/loss',
        'goal': 'minimize'
    },
    'parameters': {
        'r': {
            'values': [16, 32, 64, 128]
        },
        'lora_alpha': {
            'values': [32, 64, 128, 256]
        },
        'lora_dropout': {
            'values': [0.0, 0.05, 0.1]
        },
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-5,
            'max': 3e-3,
        },
        'target_modules': {
            # all-linear is supposedly better (thinkingmachines.ai/blog/lora/)
            # but trying both methods anyway
             'values': ['attention_only', 'all-linear']
        }
    }
}

def sweep_params():
    # Initialize the sweep run
    wandb.init()
    config = wandb.config
    config_modules = ['q_lin', 'k_lin', 'v_lin', 'out_lin'] \
        if config.target_modules == 'attention_only' else "all-linear"

    model = build_model(peft_config=LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=config.r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=config_modules
    ))

    trainer_args = SentenceTransformerTrainingArguments(
        output_dir="param_sweep_results",
        learning_rate=config.learning_rate,
        num_train_epochs=1,
        per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        eval_strategy="steps",
        eval_steps=0.5,
        logging_steps=0.2,
        report_to="wandb",     # Turn W&B logging ON
        run_name=wandb.run.name # Sync HF run name with W&B UI # pyright: ignore[reportOptionalMemberAccess]
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=trainer_args,
        train_dataset=ds['train'],
        eval_dataset=ds['test'],
        loss=MultiLabelBCEWithLogitsLoss(model)
    )
    trainer.train()


# Initialize and run the sweep experiment
# adjust count based on alloted compute time
sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT)
wandb.agent(sweep_id, function=sweep_params, count=20)


## Get best parameters

In [ ]:
api = wandb.Api()
sweep = api.sweep('ola-owo/Lyrical Miracle/sweeps/euq0smx5')
best_run = sweep.best_run()
best_config = best_run.config

In [ ]:
# values pulled from best_config above
train_config = {
    'lora_r': 128,
    'lora_alpha': 32,
    'lora_dropout': 0.1,
    'learning_rate': 2e-3,
    'target_modules': ['q_lin', 'k_lin', 'v_lin', 'out_lin']
}

# Training loop

In [ ]:
model = build_model(peft_config=LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=train_config['lora_r'],
    lora_alpha=train_config['lora_alpha'],
    lora_dropout=train_config['lora_dropout'],
    target_modules=train_config['target_modules'],
))

trainer_args = SentenceTransformerTrainingArguments(
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0.1,
    learning_rate=train_config['learning_rate'],
    metric_for_best_model='eval_loss',
    hub_revision='main',
    logging_strategy='epoch',
    eval_strategy="epoch",
    save_strategy="best",
    output_dir=LORA_MODEL_NAME,
    run_name=LORA_MODEL_NAME,  # Will be used in W&B if `wandb` is installed
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=trainer_args,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    loss=MultiLabelBCEWithLogitsLoss(model)
)
trainer.train()

# Save/Load the model

## Save

In [ ]:
# 1. Save locally
model.save_pretrained(LORA_MODEL_NAME)

# 2. Or push the entire architecture to the Hub
trainer.push_to_hub(commit_message='End of training')

## Load

In [ ]:
model = SentenceTransformer(LORA_MODEL)

# Inference

In [ ]:
test_samples = [
    "I love meeting new people and being the center of attention.",
    "I often feel anxious and worry about small things.",
    "I have a very short temper",
    "I enjoy exploring abstract concepts and complex ideas.",
    "I'm very opinionated and often debate with others",
    "I prefer intimate spaces over large crowds"
]

embeddings = model.encode(test_samples)

results_df = pl.DataFrame(embeddings, schema=BIG5_TRAITS)
results_df = results_df.insert_column(0, pl.Series("text", test_samples))
print(results_df)